# US-04-03 — PatchCore Transistor — Memory Bank complète

Expérience : EXP-001

Objectifs :
- Charger les 170 images normales Transistor du split Train.
- Extraire leurs caractéristiques avec Wide ResNet-50-2.
- Construire une Memory Bank avec KCenterGreedy.
- Sauvegarder le checkpoint.
- Vérifier sa restauration et son scoring.

Le split Test officiel n'est pas utilisé.

Source Git : exp/patchcore-transistor-exp001

In [ ]:
import os
import sys
import shutil
import subprocess
import importlib.metadata as metadata

import torch
import torchvision

print("=== ENVIRONMENT ===")
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Kaggle:", os.path.exists("/kaggle"))
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

print("\n=== DEPENDENCIES ===")

for package in ["anomalib", "timm", "lightning"]:
    try:
        print(package, metadata.version(package))
    except metadata.PackageNotFoundError:
        print(package, "NOT INSTALLED")

print("\n=== GPU ===")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No CUDA GPU detected")

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("nvidia-smi not available")

print("\n=== PRE-FLIGHT ===")

assert os.path.exists("/kaggle"), (
    "This validation must run on Kaggle"
)

assert torch.cuda.is_available(), (
    "Enable the GPU accelerator in Kaggle"
)

print("Kaggle GPU environment: OK")


In [ ]:
import sys
import json
import subprocess
from pathlib import Path

report_path = Path("/kaggle/working/patchcore_install_plan.json")

result = subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--dry-run",
        "--report", str(report_path),
        "anomalib==2.7.0"
    ],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(result.stderr[-4000:])
    raise RuntimeError("Dependency resolution failed")

report = json.loads(report_path.read_text())

planned = sorted(
    item["metadata"]["name"]
    for item in report["install"]
)

print("=== PLANNED INSTALLATIONS ===")
for name in planned:
    print(name)

critical = {
    "torch", "torchvision", "torchaudio",
    "triton"
}

changes = critical.intersection(
    name.lower().replace("_", "-")
    for name in planned
)

print("\n=== PYTORCH CHECK ===")
print("Critical packages to change:", sorted(changes))

assert not changes, (
    "Installation would modify PyTorch/CUDA packages"
)

print("\nDependency plan: OK")

In [ ]:
import sys
import subprocess

subprocess.run(
    [
        sys.executable, "-m", "pip",
        "install", "anomalib==2.7.0"
    ],
    check=True
)

print("Installation completed")

In [ ]:
import importlib.metadata as metadata
import torch
import timm
import anomalib

print("=== FINAL VERSIONS ===")

for package in [
    "torch",
    "torchvision",
    "timm",
    "anomalib",
    "lightning"
]:
    print(f"{package}: {metadata.version(package)}")

print("\n=== GPU ===")
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

assert metadata.version("anomalib") == "2.7.0"
assert torch.cuda.is_available()

print("\nPatchCore dependencies: OK")


In [ ]:
# Les versions exactes de Python, timm et Lightning
# seront enregistrées dans le rapport d'expérience.
print("Python:", sys.version.split()[0])
print("timm:", metadata.version("timm"))
print("lightning:", metadata.version("lightning"))

In [ ]:
import torch
from anomalib.models.image.patchcore.torch_model import PatchcoreModel

DEVICE = torch.device("cuda:0")
BACKBONE = "wide_resnet50_2"
LAYERS = ["layer2", "layer3"]

model = PatchcoreModel(
    backbone=BACKBONE,
    layers=LAYERS,
    pre_trained=True,
    num_neighbors=1,
).to(DEVICE)

In [ ]:
# Disable gradient computation for the feature extractor
model.feature_extractor.requires_grad_(False)

# Keep the model in evaluation mode
model.eval()

total_params = sum(
    p.numel()
    for p in model.feature_extractor.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.feature_extractor.parameters()
    if p.requires_grad
)

print("=== BACKBONE VALIDATION ===")
print("Backbone:", BACKBONE)
print("Layers:", LAYERS)
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Training mode:", model.training)

assert trainable_params == 0
assert not model.training

print("\nBackbone frozen: OK")


In [ ]:
dummy_batch = torch.randn(
    1, 3, 256, 256,
    device=DEVICE
)

with torch.inference_mode():
    features = model.feature_extractor(dummy_batch)

print("=== FEATURE EXTRACTION ===")

for layer in LAYERS:
    assert layer in features, f"Missing layer: {layer}"

    feature_map = features[layer]

    print(f"\n{layer}")
    print("Shape:", tuple(feature_map.shape))
    print("Dtype:", feature_map.dtype)
    print("Device:", feature_map.device)
    print("Finite:", torch.isfinite(feature_map).all().item())

    assert feature_map.ndim == 4
    assert feature_map.device.type == "cuda"
    assert torch.isfinite(feature_map).all()

print("\nFeature extraction: OK")

### Données et construction de la Memory Bank — Transistor

Les 170 images normales du split Train alimentent la banque mémoire.

Le backbone Wide ResNet-50-2 extrait des vecteurs de caractéristiques
pour chaque image.

KCenterGreedy sélectionne 1 % de ces vecteurs afin de construire
une Memory Bank représentative.

Les 43 images de validation sont réservées au scoring.
Le split Test officiel n'est pas utilisé.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/MRdrissi/industrial-defect-ai.git"
REPO_DIR = Path("/kaggle/working/industrial-defect-ai")
SOURCE_TAG = "exp/patchcore-transistor-exp001"

# Cloner le dépôt si nécessaire
if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)],
        check=True
    )

# Récupérer le tag expérimental
subprocess.run(
    ["git", "-C", str(REPO_DIR), "fetch", "origin", "--tags"],
    check=True
)

# Se positionner sur le commit exact de l'expérience
subprocess.run(
    [
        "git", "-C", str(REPO_DIR),
        "checkout", "--detach", f"refs/tags/{SOURCE_TAG}"
    ],
    check=True
)

source_commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True
).strip()

sys.path.insert(0, str(REPO_DIR))

# Charger la configuration de l'expérience
config_path = (
        REPO_DIR / "experiments/configs/patchcore_transistor.json"
)

config = json.loads(config_path.read_text(encoding="utf-8"))

assert config["experiment_id"] == "EXP-001"
assert config["memory_bank"]["sample_count"] == 170
assert config["category"] == "transistor"
assert config["model"]["backbone"] == BACKBONE
assert config["model"]["layers"] == LAYERS
assert config["model"]["num_neighbors"] == 1
assert config["memory_bank"]["sampling"] == "all"
assert config["memory_bank"]["sampling_ratio"] == 0.01

from src.data.mvtec_dataset import MVTecDataset
from src.data.transforms import MVTecTransform
from src.data.dataloaders import create_dataloader

print("Git commit:", source_commit)
print("Git tag:", SOURCE_TAG)
print("Configuration:", config["experiment_id"])
print("Imports et configuration : OK")

In [ ]:
import os
import torch
from pathlib import Path

# Locate MVTec dataset root
dataset_root = None

for current, dirs, _ in os.walk("/kaggle/input"):
    candidate = Path(current)

    if "transistor" in dirs and (
        candidate / "transistor" / "train" / "good"
    ).is_dir():
        dataset_root = candidate
        break

assert dataset_root is not None, (
    "MVTec AD dataset not found. Add it as Kaggle Input."
)

manifest = (
    REPO_DIR / "docs/data/us-02-03/transistor_manifest.csv"
)

transform = MVTecTransform(
    size=tuple(config["transform"]["size"]),
    profile=config["transform"]["profile"]
)

train_dataset = MVTecDataset(
    dataset_root=dataset_root,
    manifest_path=manifest,
    category="transistor",
    split="train",
    transform=transform
)

train_loader = create_dataloader(
    train_dataset,
    batch_size=config["memory_bank"]["batch_size"],
    num_workers=config["memory_bank"]["num_workers"],
    seed=config["memory_bank"]["seed"]
)

batch = next(iter(train_loader))
images = batch["image"].to(DEVICE)

print("=== REAL TRANSISTOR BATCH ===")
print("Dataset root:", dataset_root)
print("Train samples:", len(train_dataset))
print("Image shape:", tuple(images.shape))
print("Dtype:", images.dtype)
print("Min:", images.min().item())
print("Max:", images.max().item())
print("Labels:", batch["label"].tolist())
print("Paths:", batch["image_path"])

assert len(train_dataset) == 170
assert images.shape == (2, 3, 256, 256)
assert torch.isfinite(images).all()
assert torch.all(batch["label"] == 0)

print("\nMVTec Transistor pipeline: OK")

In [ ]:
with torch.inference_mode():
    real_features = model.feature_extractor(images)

print("=== REAL TRANSISTOR FEATURES ===")

for layer in LAYERS:
    tensor = real_features[layer]

    print(layer, tuple(tensor.shape))
    assert torch.isfinite(tensor).all()

print("\nReal-image feature extraction: OK")


In [ ]:
# US-04-03 — Utiliser toutes les images normales Transistor

SEED = config["memory_bank"]["seed"]
NUM_IMAGES = len(train_dataset)

assert NUM_IMAGES == 170
assert NUM_IMAGES == config["memory_bank"]["sample_count"]
assert config["memory_bank"]["sampling"] == "all"

print("=== PATCHCORE TRANSISTOR TRAIN ===")
print("Category:", config["category"])
print("Train images:", NUM_IMAGES)
print("Seed:", SEED)
print("Batch size:", config["memory_bank"]["batch_size"])

print("\nFull training dataset: OK")

In [ ]:
# US-04-03 — Extract patch embeddings from all 170 images

model.train()
model.feature_extractor.eval()

patch_vectors = []
selected_paths = []
images_seen = 0

with torch.inference_mode():
    for batch_idx, batch in enumerate(train_loader):
        images_batch = batch["image"].to(DEVICE)

        assert torch.all(batch["label"] == 0)

        embeddings = model(images_batch)

        if batch_idx == 0:
            print("First batch shape:", tuple(embeddings.shape))

        assert embeddings.ndim == 2
        assert embeddings.shape[0] == images_batch.shape[0] * 32 * 32
        assert embeddings.shape[1] == 1536
        assert torch.isfinite(embeddings).all()

        # Stocker sur CPU pour limiter l'utilisation de la VRAM
        patch_vectors.append(embeddings.detach().cpu())

        selected_paths.extend(batch["image_path"])
        images_seen += images_batch.shape[0]

all_embeddings = torch.cat(patch_vectors, dim=0)
del patch_vectors

expected_vectors = NUM_IMAGES * 32 * 32

print("\n=== PATCH EMBEDDINGS ===")
print("Images processed:", images_seen)
print("Embedding matrix:", tuple(all_embeddings.shape))
print("Dtype:", all_embeddings.dtype)
print("Device:", all_embeddings.device)
print("Finite:", torch.isfinite(all_embeddings).all().item())

assert images_seen == 170
assert len(selected_paths) == 170
assert len(set(selected_paths)) == 170
assert all("/train/good/" in p for p in selected_paths)
assert all_embeddings.shape == (expected_vectors, 1536)
assert torch.isfinite(all_embeddings).all()

print("\nFull Transistor embeddings: OK")

In [ ]:
# US-04-03 — Build full Transistor Memory Bank

import torch
from anomalib.models.components import KCenterGreedy

CORESET_RATIO = config["memory_bank"]["sampling_ratio"]

torch.manual_seed(SEED)

embedding_shape = list(all_embeddings.shape)

print("=== CORESET SAMPLING ===")
print("Original embeddings:", embedding_shape)
print("Sampling ratio:", CORESET_RATIO)

# Transferer les embeddings sur GPU
gpu_embeddings = all_embeddings.to(DEVICE)

# Selectionner les patches representatifs
sampler = KCenterGreedy(
    embedding=gpu_embeddings,
    sampling_ratio=CORESET_RATIO
)

coreset = sampler.sample_coreset()

# Conserver les vecteurs selectionnes
model.memory_bank = coreset.detach().contiguous()

# Basculer en mode inference
model.eval()

bank = model.memory_bank

expected_bank_size = int(
    embedding_shape[0] * CORESET_RATIO
)

print("\n=== MEMORY BANK ===")
print("Bank shape:", tuple(bank.shape))
print("Device:", bank.device)
print("Dtype:", bank.dtype)
print("Finite:", torch.isfinite(bank).all().item())

assert bank.ndim == 2
assert bank.shape == (expected_bank_size, 1536)
assert torch.isfinite(bank).all()
assert not model.training

# Liberer les references temporaires
del sampler, coreset, gpu_embeddings, all_embeddings
torch.cuda.empty_cache()

print("\nFull Transistor Memory Bank: OK")

In [ ]:
val_dataset = MVTecDataset(
    dataset_root=dataset_root,
    manifest_path=manifest,
    category="transistor",
    split="val",
    transform=transform
)

assert len(val_dataset) == config["inference"]["expected_samples"]

sample = val_dataset[0]

assert sample["image_path"] == config["inference"]["image_path"]

assert int(sample["label"]) == 0

val_image = sample["image"].unsqueeze(0).to(DEVICE)

with torch.inference_mode():
    prediction = model(val_image)

score = prediction.pred_score
anomaly_map = prediction.anomaly_map

print("=== FIRST PATCHCORE INFERENCE ===")
print("Image:", sample["image_path"])
print("Input:", tuple(val_image.shape))
print("Score:", score.detach().cpu().tolist())
print("Anomaly map:", tuple(anomaly_map.shape))
print("Map min:", anomaly_map.min().item())
print("Map max:", anomaly_map.max().item())

assert score.numel() == 1
assert torch.isfinite(score).all()
assert torch.isfinite(anomaly_map).all()
assert anomaly_map.shape[-2:] == (256, 256)

print("\nPatchCore inference: OK")

In [ ]:
# US-04-03 — Save model and experiment report

from pathlib import Path
import hashlib
import json
import importlib.metadata as metadata

RUN_DIR = Path(
    "/kaggle/working/patchcore/transistor/EXP-001"
)
RUN_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = RUN_DIR / "patchcore_model.pt"

state = {
    key: tensor.detach().cpu().contiguous()
    for key, tensor in model.state_dict().items()
}

assert "memory_bank" in state
assert tuple(state["memory_bank"].shape) == tuple(bank.shape)

torch.save(
    {
        "model_state_dict": state,
        "backbone": BACKBONE,
        "layers": LAYERS,
        "num_neighbors": config["model"]["num_neighbors"]
    },
    checkpoint_path
)

checkpoint_sha256 = hashlib.sha256(
    checkpoint_path.read_bytes()
).hexdigest()

report = {
    "run_id": "patchcore-transistor-exp001",
    "experiment_id": "EXP-001",
    "git_commit": source_commit,
    "git_tag": SOURCE_TAG,
    "config_sha256": hashlib.sha256(
        config_path.read_bytes()
    ).hexdigest(),
    "manifest_sha256": hashlib.sha256(
        manifest.read_bytes()
    ).hexdigest(),
    "versions": {
        package: metadata.version(package)
        for package in [
            "torch", "torchvision",
            "anomalib", "timm", "lightning"
        ]
    },
    "backbone": BACKBONE,
    "layers": LAYERS,
    "seed": SEED,
    "train_images": selected_paths,
    "train_image_count": NUM_IMAGES,
    "coreset_ratio": CORESET_RATIO,
    "embedding_shape": embedding_shape,
    "memory_bank_shape": list(bank.shape),
    "validation_image": sample["image_path"],
    "anomaly_score": float(score.item()),
    "anomaly_map_shape": list(anomaly_map.shape),
    "checkpoint_sha256": checkpoint_sha256
}

report_path = RUN_DIR / "memory_bank_report.json"

report_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("=== EXP-001 ARTIFACTS ===")
print("Checkpoint:", checkpoint_path)
print("SHA256:", checkpoint_sha256)
print("Git commit:", source_commit)
print("Report:", report_path)
print("Artifacts saved: OK")

In [ ]:

from anomalib.models.image.patchcore.torch_model import PatchcoreModel

saved = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

restored_model = PatchcoreModel(
    backbone=saved["backbone"],
    layers=saved["layers"],
    pre_trained=False,
    num_neighbors=saved["num_neighbors"]
)

restored_model.load_state_dict(
    saved["model_state_dict"],
    strict=True
)

assert tuple(restored_model.memory_bank.shape) == tuple(bank.shape)

restored_model = restored_model.to(DEVICE)
restored_model.feature_extractor.requires_grad_(False)
restored_model.eval()

with torch.inference_mode():
    restored_prediction = restored_model(val_image)

torch.testing.assert_close(
    restored_prediction.pred_score,
    score,
    rtol=1e-5,
    atol=1e-4
)

torch.testing.assert_close(
    restored_prediction.anomaly_map,
    anomaly_map,
    rtol=1e-5,
    atol=1e-4
)

print("=== RESTORATION CHECK ===")
print("Memory Bank:", tuple(restored_model.memory_bank.shape))
print("Original score:", score.item())
print("Restored score:", restored_prediction.pred_score.item())
print("Score and anomaly map: MATCH")
print("\nRestoration: OK")


In [ ]:

import matplotlib.pyplot as plt
import numpy as np

# Reverse ImageNet normalization for visualization only
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

image_rgb = (
    sample["image"].cpu() * std + mean
).clamp(0, 1).permute(1, 2, 0).numpy()

heatmap = anomaly_map[0, 0].detach().cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(image_rgb)
axes[0].set_title("Original Transistor")

axes[1].imshow(heatmap, cmap="magma")
axes[1].set_title("Raw anomaly map")

axes[2].imshow(image_rgb)
axes[2].imshow(heatmap, cmap="magma", alpha=0.45)
axes[2].set_title("Overlay")

for ax in axes:
    ax.axis("off")

fig.suptitle(
    f"PatchCore — Raw score: {score.item():.4f}"
)
fig.tight_layout()

fig.savefig(
    RUN_DIR / "anomaly_visualization.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()


In [ ]:

import numpy as np

reference_path = RUN_DIR / "reference_prediction.npz"

np.savez_compressed(
    reference_path,
    pred_score=score.detach().cpu().numpy(),
    anomaly_map=anomaly_map.detach().cpu().numpy()
)

print("Reference:", reference_path)
print("Score:", float(score.item()))
print("Map shape:", tuple(anomaly_map.shape))
print("Reference saved: OK")
